In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from workflow.scripts.utils import global_avg,read_list_input_paths
from workflow.scripts.plotting_tools import create_facet_plot

In [ ]:
if snakemake.rule == 'plot_surface_toa_albedo':
    rsdt = read_list_input_paths(snakemake.input.rsdt)[0]
    rsutcsaf = read_list_input_paths(snakemake.input.rsutcsaf)[0]

    rsdt = {m: rsdt[m].isel(time=slice(1,None)).mean(dim='time') for m in rsdt}
    rsutcsaf = {m: rsutcsaf[m].isel(time=slice(1,None)).mean(dim='time') for m in rsutcsaf}

    rsutcsaf['NorESM2-LM'] = rsutcsaf['NorESM2-LM'].assign(rsutcsaf = rsdt['NorESM2-LM']['rsdt']-rsutcsaf['NorESM2-LM']['rsutcsaf'])

    def calc_albedo(rsdt, rsutcsaf):
        return rsutcsaf['rsutcsaf']/rsdt['rsdt']
    surfal = {m :  calc_albedo(rsdt[m], rsutcsaf[m]) for m in rsdt}
else:
    rsds = read_list_input_paths(snakemake.input.rsds)[0]
    rsus = read_list_input_paths(snakemake.input.rsus)[0]

    rsds = {m: rsds[m].isel(time=slice(1,None)).mean(dim='time') for m in rsds}
    rsus = {m: rsus[m].isel(time=slice(1,None)).mean(dim='time') for m in rsus}
    def calc_albedo(rsds, rsus):
        return rsus['rsus']/rsds['rsds']
    
    surfal = {m :  calc_albedo(rsds[m], rsus[m]) for m in rsds}

In [ ]:
fig, axes, cax=create_facet_plot(len(surfal), subplot_kw={'projection':ccrs.EckertIV()})

cmap = mpl.cm.viridis.resampled(17)
for m,a in zip(surfal,axes):
    surfal[m].plot(ax=axes[a], transform=ccrs.PlateCarree(), cmap=cmap, vmin=0, vmax=0.8, add_colorbar=False)
    axes[a].set_title(m)
    axes[a].coastlines()
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0.2,0.8),cmap=cmap), cax=cax, extend='both')
cbar.set_label("Surface Albedo (Unitless)")
plt.savefig(snakemake.output[0], dpi=300, bbox_inches='tight')